# Section 4: Hand work to the right specialist

Goal: a triage agent reads each customer email and hands it to a tracking or a billing specialist,
each with its own tool. We check the routing on the whole inbox, fix what the tools return,
and let a conversation remember the previous question.

## Step 1. Connect, and recall the data

In [1]:
import json, os, sqlite3
from dotenv import load_dotenv
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

load_dotenv()
retry = RetryConfig("backoff", BackoffStrategy(1000, 20000, 2.0, 120000),
                    retry_connection_errors=True)
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"], retry_config=retry,
                 timeout_ms=120000)
MODEL = "mistral-medium-latest"

db = sqlite3.connect("dock/dock.sqlite")
inbox = json.loads(open("dock/inbox.json", encoding="utf-8").read())
truth = json.loads(open("dock/inbox_truth.json").read())
print(len(inbox), "emails ·", sorted({t["intent"] for t in truth.values()}))

20 emails · ['booking', 'customs', 'damage', 'invoice', 'other', 'tracking']


## Step 2. Two tools, one per specialist

In [2]:
def get_shipment(shipment_id: str) -> str:
    """Look up a shipment (like HF-26001): status, ports and ETA."""
    row = db.execute("SELECT id, status, origin_port, destination_port, eta "
                     "FROM shipments WHERE id = ?", (shipment_id,)).fetchone()
    return json.dumps(row) if row else f"No shipment {shipment_id}"


def get_invoice(shipment_id: str) -> str:
    """Look up the invoice of a shipment (like HF-26001): amounts and total."""
    row = db.execute("SELECT invoice_no, subtotal, freight, insurance, total "
                     "FROM invoices WHERE shipment_id = ?", (shipment_id,)).fetchone()
    return json.dumps(row) if row else f"No invoice for {shipment_id}"

print(get_invoice("HF-26040"))

["INV-2026-3139", 3630.0, 217.8, 14.52, 3862.32]


## Step 3. Create the agents

In [3]:
from mistralai.extra.run.context import RunContext

def tools_for(*functions):
    ctx = RunContext(model=MODEL)
    for f in functions:
        ctx.register_func(f)
    return ctx.get_tools()

tracking = client.beta.agents.create(
    model=MODEL, name="dock-tracking",
    description="Answers where a shipment is and when it arrives.",
    instructions="You answer shipment tracking questions. Use get_shipment.",
    tools=tools_for(get_shipment))
billing = client.beta.agents.create(
    model=MODEL, name="dock-billing",
    description="Answers questions about invoices and charges.",
    instructions="You answer invoice questions. Use get_invoice.",
    tools=tools_for(get_invoice))
print(tracking.id[:12], tracking.name, "|", billing.id[:12], billing.name)

ag_01a185e5d dock-tracking | ag_01a185e5d dock-billing


In [4]:
triage = client.beta.agents.create(
    model=MODEL, name="dock-triage",
    description="Front desk of Harbor Freight Co.",
    instructions="Hand tracking questions to dock-tracking and invoice questions to "
                 "dock-billing. For anything else, do not hand off: reply in one "
                 "sentence that a person from operations will follow up.")
triage = client.beta.agents.update(agent_id=triage.id,
                                   handoffs=[tracking.id, billing.id])
print(triage.name, "can hand off to", len(triage.handoffs), "agents")

dock-triage can hand off to 2 agents


## Step 4. One request, step by step

In [5]:
NAMES = {tracking.id: "tracking", billing.id: "billing", triage.id: "triage"}

async def run(text, conversation_id=None):
    async with RunContext(agent_id=triage.id) as ctx:
        ctx.conversation_id = conversation_id
        ctx.register_func(get_shipment)
        ctx.register_func(get_invoice)
        return await client.beta.conversations.run_async(run_ctx=ctx, inputs=text)


def show(result):
    for e in result.output_entries:
        kind = type(e).__name__.replace("Entry", "")
        who = NAMES.get(getattr(e, "agent_id", None), "")
        if kind == "AgentHandoff":
            print(f"{kind:14} -> {e.next_agent_name}")
        elif kind == "FunctionCall":
            print(f"{kind:14} {who:9} {e.name}({e.arguments})")
        elif kind == "MessageOutput":
            print(f"{kind:14} {who:9} {str(e.content)[:200]}")

result = await run(inbox[2]["body"])
show(result)

AgentHandoff   -> dock-billing
FunctionCall   billing   get_invoice({"shipment_id": "HF-26040"})
MessageOutput  billing   The invoice **INV-2026-3139** for shipment **HF-26040** shows the following breakdown:
- **Subtotal**: 3,630.00 USD
- **Shipping**: 217.80 USD
- **Taxes/Fees**: 14.52 USD
- **Total**: **3,862.32 USD**


## Step 5. Fix what the tool returns

In [6]:
def get_invoice(shipment_id: str) -> str:
    """Look up the invoice of a shipment (like HF-26001): amounts and total."""
    cur = db.execute("SELECT invoice_no, subtotal, freight, insurance, total "
                     "FROM invoices WHERE shipment_id = ?", (shipment_id,))
    row = cur.fetchone()
    if row is None:
        return f"No invoice for {shipment_id}"
    names = [col[0] for col in cur.description]
    return json.dumps(dict(zip(names, row)))


def get_shipment(shipment_id: str) -> str:
    """Look up a shipment (like HF-26001): status, ports and ETA."""
    cur = db.execute("SELECT id, status, origin_port, destination_port, eta "
                     "FROM shipments WHERE id = ?", (shipment_id,))
    row = cur.fetchone()
    if row is None:
        return f"No shipment {shipment_id}"
    return json.dumps(dict(zip([col[0] for col in cur.description], row)))

print(get_invoice("HF-26040"))
result = await run(inbox[2]["body"])
show(result)

{"invoice_no": "INV-2026-3139", "subtotal": 3630.0, "freight": 217.8, "insurance": 14.52, "total": 3862.32}


AgentHandoff   -> dock-billing
FunctionCall   billing   get_invoice({"shipment_id": "HF-26040"})
MessageOutput  billing   The invoice **INV-2026-3139** for shipment **HF-26040** shows the following details:

- **Subtotal**: **3,630.00 USD**
- **Freight**: **217.80 USD**
- **Insurance**: **14.52 USD**
- **Total**: **3,862


## Step 6. Route the whole inbox

In [7]:
ROUTE = {"tracking": "tracking", "invoice": "billing"}   # everything else stays with triage

routes = {}
for mail in inbox:
    result = await run(mail["subject"] + "\n\n" + mail["body"])
    handoffs = [e.next_agent_name for e in result.output_entries
                if type(e).__name__ == "AgentHandoffEntry"]
    routes[mail["id"]] = handoffs[-1].removeprefix("dock-") if handoffs else "triage"

wrong = {mid: (got, ROUTE.get(truth[mid]["intent"], "triage"))
         for mid, got in routes.items() if got != ROUTE.get(truth[mid]["intent"], "triage")}
print("routed correctly:", 20 - len(wrong), "/ 20")
for mid, (got, want) in wrong.items():
    print(" ", mid, truth[mid]["intent"], "went to", got, "- expected", want)

routed correctly: 17 / 20
  M02 customs went to tracking - expected triage
  M12 customs went to tracking - expected triage
  M13 damage went to tracking - expected triage


## Step 7. Fix the routing rule

In [8]:
triage = client.beta.agents.update(
    agent_id=triage.id,
    instructions="Hand off ONLY two kinds of request. Where a shipment is or when it "
                 "arrives: dock-tracking. Invoice amounts and charges: dock-billing. "
                 "Customs documents, damage, bookings and anything else: do not hand off, "
                 "even if a shipment is mentioned. Reply in one sentence that a person "
                 "from operations will follow up.")

routes = {}
for mail in inbox:
    result = await run(mail["subject"] + "\n\n" + mail["body"])
    handoffs = [e.next_agent_name for e in result.output_entries
                if type(e).__name__ == "AgentHandoffEntry"]
    routes[mail["id"]] = handoffs[-1].removeprefix("dock-") if handoffs else "triage"

wrong = {mid: got for mid, got in routes.items()
         if got != ROUTE.get(truth[mid]["intent"], "triage")}
print("routed correctly:", 20 - len(wrong), "/ 20", wrong or "")

routed correctly: 19 / 20 {'M19': 'triage'}


## Step 8. A conversation remembers

In [9]:
first = await run("Where is HF-26012 right now?")
show(first)
print()
second = await run("And the other one, HF-26016?", conversation_id=first.conversation_id)
show(second)
print()
fresh = await run("And the other one?")
show(fresh)

AgentHandoff   -> dock-tracking
FunctionCall   tracking  get_shipment({"shipment_id": "HF-26012"})
MessageOutput  tracking  The shipment **HF-26012** is currently **at the origin port**, **Yokohama (JPYOK)**. Its destination is **Felixstowe (GBFXT)**, with an estimated arrival date of **October 21, 2026**.

Would you like 



FunctionCall   tracking  get_shipment({"shipment_id": "HF-26016"})
MessageOutput  tracking  The shipment **HF-26016** is currently **on vessel**, en route from **Yokohama (JPYOK)** to **Felixstowe (GBFXT)**. Its estimated arrival date is **today, October 29, 2026**.

Would you like to track 



MessageOutput  triage    A person from operations will follow up.


## Step 9. Clean up

In [10]:
for agent in (triage, tracking, billing):
    client.beta.agents.delete(agent_id=agent.id)
print("deleted:", triage.name, tracking.name, billing.name)

deleted: dock-triage dock-tracking dock-billing
